## 1. Clone Dataset

In [2]:
!git clone https://github.com/Franck-Dernoncourt/pubmed-rct.git
!ls pubmed-rct

fatal: destination path 'pubmed-rct' already exists and is not an empty directory.
'ls' is not recognized as an internal or external command,
operable program or batch file.


In [3]:
!ls pubmed-rct/PubMed_20k_RCT_numbers_replaced_with_at_sign

'ls' is not recognized as an internal or external command,
operable program or batch file.


In [4]:
data_dir = "pubmed-rct/PubMed_20k_RCT_numbers_replaced_with_at_sign/"

## 2. Imports

In [7]:
import tensorflow as tf
import numpy as np
import pandas as pd
import os
import json
import string
import random
import matplotlib.pyplot as plt
from tensorflow.keras import layers
from tensorflow.keras.layers import TextVectorization
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report
print('TF version:', tf.__version__)

TF version: 2.16.1


## 3. Data Preprocessing

In [8]:
def get_lines(filename):
    with open(filename, 'r') as f:
        return f.readlines()

"""def preprocess_text_lines(filename):
    abstract_samples = []
    abstract_lines = ''
    input_lines = get_lines(filename)
    for line in input_lines:
        if line.startswith('###'):
            abstract_id = line
            abstract_lines = ''
        elif line.isspace():
            abstract_line_split = abstract_lines.splitlines()
            for abstract_line_no, abstract_line in enumerate(abstract_line_split):
                line_data = {}
                target_text_split = abstract_line.split('\t')
                line_data['target'] = target_text_split[0]
                line_data['text'] = target_text_split[1].lower()
                line_data['line_number'] = abstract_line_no
                line_data['total_lines'] = len(abstract_line_split) - 1
                abstract_samples.append(line_data)
        else:
            abstract_lines += line
    return abstract_samples
"""

"def preprocess_text_lines(filename):\n    abstract_samples = []\n    abstract_lines = ''\n    input_lines = get_lines(filename)\n    for line in input_lines:\n        if line.startswith('###'):\n            abstract_id = line\n            abstract_lines = ''\n        elif line.isspace():\n            abstract_line_split = abstract_lines.splitlines()\n            for abstract_line_no, abstract_line in enumerate(abstract_line_split):\n                line_data = {}\n                target_text_split = abstract_line.split('\t')\n                line_data['target'] = target_text_split[0]\n                line_data['text'] = target_text_split[1].lower()\n                line_data['line_number'] = abstract_line_no\n                line_data['total_lines'] = len(abstract_line_split) - 1\n                abstract_samples.append(line_data)\n        else:\n            abstract_lines += line\n    return abstract_samples\n"

In [9]:
def preprocess_text_lines(filename):
  abstract_samples=[]
  abstract_lines=""
  input_lines=get_lines(filename)
  for line in input_lines:
    if line.startswith("###"):
      abstract_id=line
      abstract_lines=""
    elif line.isspace():
      abstract_line_split=abstract_lines.splitlines()
      for abstract_line_no,abstract_line in enumerate(abstract_line_split):
        line_data={}
        split_sentence=abstract_line.split("\t")
        label=split_sentence[0]
        sentence=split_sentence[1:]
        line_data["target"]=label
        line_data["text"]=str(sentence).lower()
        line_data["line_number"]=abstract_line_no
        line_data["total_lines"]=len(abstract_line_split)
        abstract_samples.append(line_data)
    else:
      abstract_lines+=line

  return abstract_samples
train_samples = preprocess_text_lines(data_dir + 'train.txt')
val_samples   = preprocess_text_lines(data_dir + 'dev.txt')
test_samples  = preprocess_text_lines(data_dir + 'test.txt')
print(f'Train: {len(train_samples)} | Val: {len(val_samples)} | Test: {len(test_samples)}')

Train: 180040 | Val: 30212 | Test: 30135


In [10]:
train_df = pd.DataFrame(train_samples)
val_df   = pd.DataFrame(val_samples)
test_df  = pd.DataFrame(test_samples)
train_df.head()

,target,text,line_number,total_lines
0,OBJECTIVE,['to investigate the efficacy of @ weeks of da...,0,12
1,METHODS,['a total of @ patients with primary knee oa w...,1,12
2,METHODS,['outcome measures included pain reduction and...,2,12
3,METHODS,['pain was assessed using the visual analog pa...,3,12
4,METHODS,['secondary outcome measures included the west...,4,12


In [11]:
train_sentences = train_df['text'].tolist()
val_sentences   = val_df['text'].tolist()
test_sentences  = test_df['text'].tolist()

## 4. Positional Features (One-Hot)

In [12]:
# line_number depth=15 (covers 95th percentile)
train_line_numbers_one_hot = tf.one_hot(train_df['line_number'].to_numpy(), depth=15)
val_line_numbers_one_hot   = tf.one_hot(val_df['line_number'].to_numpy(),   depth=15)
test_line_numbers_one_hot  = tf.one_hot(test_df['line_number'].to_numpy(),  depth=15)

# total_lines depth=20 (covers 97th percentile)
total_line_percentile = (np.percentile(train_df['total_lines'], 97))  # = 20
#total_line_percentile = 20

#total_line_percentile=np.percentile(train_df["total_lines"],97)

train_total_lines = tf.one_hot(train_df['total_lines'].to_numpy(), depth=total_line_percentile)
val_total_lines   = tf.one_hot(val_df['total_lines'].to_numpy(),   depth=total_line_percentile)
test_total_lines  = tf.one_hot(test_df['total_lines'].to_numpy(),  depth=total_line_percentile)

print('line_number one-hot shape:', train_line_numbers_one_hot.shape)
print('total_lines one-hot shape:', train_total_lines.shape)

line_number one-hot shape: (180040, 15)
total_lines one-hot shape: (180040, 20)


## 5. Character-Level Features

In [13]:
def split_chars(text):
    return ' '.join(list(text))

train_chars = [split_chars(s) for s in train_sentences]
val_chars   = [split_chars(s) for s in val_sentences]
test_chars  = [split_chars(s) for s in test_sentences]

# 95th percentile character length
char_lens = [len(s) for s in train_sentences]
output_seq_char_len = int(np.percentile(char_lens, 95))  # = 294
print('Char seq length (95th pct):', output_seq_char_len)

# Character vocabulary
alphabet = string.ascii_lowercase + string.digits
NUM_CHAR_TOKENS = len(alphabet) + 2
char_vectorizer = TextVectorization(max_tokens=NUM_CHAR_TOKENS,
                                    output_sequence_length=output_seq_char_len)
char_vectorizer.adapt(train_chars)
char_vocab = char_vectorizer.get_vocabulary()
char_embed = layers.Embedding(input_dim=len(char_vocab), output_dim=25,
                              mask_zero=True, name='char_embed')
print('Char vocab size:', len(char_vocab))

Char seq length (95th pct): 294
Char vocab size: 28


## 6. Label Encoding

In [14]:
label_encode = LabelEncoder()
train_labels_encoded = label_encode.fit_transform(train_df['target'].to_numpy())
val_labels_encoded   = label_encode.fit_transform(val_df['target'].to_numpy())
test_labels_encoded  = label_encode.fit_transform(test_df['target'].to_numpy())

num_classes  = len(label_encode.classes_)
class_names  = list(label_encode.classes_)
print('Classes:', class_names)

# One-hot for training (CategoricalCrossentropy)
one_hot_encoder = OneHotEncoder(sparse_output=False)
train_labels_one_hot = one_hot_encoder.fit_transform(train_df['target'].to_numpy().reshape(-1, 1))
val_labels_one_hot   = one_hot_encoder.transform(val_df['target'].to_numpy().reshape(-1, 1))
test_labels_one_hot  = one_hot_encoder.transform(test_df['target'].to_numpy().reshape(-1, 1))

Classes: ['BACKGROUND', 'CONCLUSIONS', 'METHODS', 'OBJECTIVE', 'RESULTS']


## 7. Token Vectorizer

In [15]:
MAX_TOKENS = 20000
MAX_LEN    = 50
EMBED_DIM  = 128

token_vectorizer = layers.TextVectorization(
    max_tokens=MAX_TOKENS,
    output_sequence_length=MAX_LEN,
    standardize='lower_and_strip_punctuation'
)
token_vectorizer.adapt(train_sentences)
print('Token vocab size:', len(token_vectorizer.get_vocabulary()))

Token vocab size: 20000


## 8. Model 5 — Tribrid Architecture (Token + Char + Positional)

In [16]:
# ── Token model (BiLSTM) ────────────────────────────────────────────────
token_input  = layers.Input(shape=(1,), dtype=tf.string, name='token_input')
x            = token_vectorizer(token_input)
x            = layers.Embedding(input_dim=MAX_TOKENS, output_dim=EMBED_DIM, mask_zero=True)(x)
x            = layers.Bidirectional(layers.LSTM(64))(x)
token_output = layers.Dense(128, activation='relu')(x)
token_model  = tf.keras.Model(inputs=token_input, outputs=token_output)

# ── Char model (Conv1D) ──────────────────────────────────────────────────
char_input  = layers.Input(shape=(1,), dtype=tf.string, name='char_input')
x           = char_vectorizer(char_input)
x           = char_embed(x)
x           = layers.Conv1D(64, kernel_size=5, activation='relu', padding='same')(x)
x           = layers.GlobalMaxPooling1D()(x)
char_output = layers.Dense(128, activation='relu')(x)
char_model  = tf.keras.Model(inputs=char_input, outputs=char_output)

# ── Line number model ────────────────────────────────────────────────────
line_number_input  = layers.Input(shape=(15,), dtype=tf.float32, name='line_number_input')
x                  = layers.Dense(32, activation='relu')(line_number_input)
line_number_output = layers.Dense(32, activation='relu')(x)
line_number_model  = tf.keras.Model(inputs=line_number_input, outputs=line_number_output)

# ── Total lines model ────────────────────────────────────────────────────
total_line_input  = layers.Input(shape=(20,), dtype=tf.float32, name='total_line_input')
x                 = layers.Dense(32, activation='relu')(total_line_input)
total_line_output = layers.Dense(32, activation='relu')(x)
total_line_model  = tf.keras.Model(inputs=total_line_input, outputs=total_line_output)

# ── Combine all ──────────────────────────────────────────────────────────
combined     = layers.Concatenate()([line_number_model.output,
                                     total_line_model.output,
                                     token_model.output,
                                     char_model.output])
x            = layers.Dense(256, activation='relu')(combined)
x            = layers.Dropout(0.2)(x)
output_layer = layers.Dense(num_classes, activation='softmax')(x)

model_5 = tf.keras.Model(
    inputs=[line_number_model.input, total_line_model.input,
            token_model.input, char_model.input],
    outputs=output_layer
)
model_5.summary()

c:\Users\NARAYAN KRISHNA\OneDrive\Documents\Skimlit\dl\Lib\site-packages\keras\src\layers\layer.py:982: UserWarning: Layer 'conv1d' (of type Conv1D) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(


Model: "functional_4"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ char_input          │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ token_input         │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ text_vectorization  │ (None, 294)       │          0 │ char_input[0][0]  │
│ (TextVectorization) │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ text_vectorization… │ (None, 50)        │          0 │ token_input[0][0] │
│ (TextVectorization) │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ char_embed          │ (None, 294, 25)   │        700 │ text_vectorizati… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ line_number_input   │ (None, 15)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ total_line_input    │ (None, 20)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 50, 128)   │  2,560,000 │ text_vectorizati… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal           │ (None, 50)        │          0 │ text_vectorizati… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d (Conv1D)     │ (None, 294, 64)   │      8,064 │ char_embed[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 32)        │        512 │ line_number_inpu… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_4 (Dense)     │ (None, 32)        │        672 │ total_line_input… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional       │ (None, 128)       │     98,816 │ embedding[0][0],  │
│ (Bidirectional)     │                   │            │ not_equal[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_max_pooling… │ (None, 64)        │          0 │ conv1d[0][0]      │
│ (GlobalMaxPooling1… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 32)        │      1,056 │ dense_2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_5 (Dense)     │ (None, 32)        │      1,056 │ dense_4[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 128)       │     16,512 │ bidirectional[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 128)       │      8,320 │ global_max_pooli… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 320)       │          0 │ dense_3[0][0],    │
│ (Concatenate)       │                   │            │ dense_5[0][0],  

 Total params: 2,779,169 (10.60 MB)

 Trainable params: 2,779,169 (10.60 MB)

 Non-trainable params: 0 (0.00 B)

## 9. Build tf.data Datasets

In [17]:
BATCH_SIZE = 32

# FIX: variable names are *_data (not *_dataset) — consistent throughout
train_pos_char_token_data = tf.data.Dataset.from_tensor_slices((
    (train_line_numbers_one_hot, train_total_lines, train_sentences, train_chars),
    train_labels_one_hot
)).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

val_pos_char_token_data = tf.data.Dataset.from_tensor_slices((
    (val_line_numbers_one_hot, val_total_lines, val_sentences, val_chars),
    val_labels_one_hot
)).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

test_pos_char_token_data = tf.data.Dataset.from_tensor_slices((
    (test_line_numbers_one_hot, test_total_lines, test_sentences, test_chars),
    test_labels_one_hot
)).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

print('Train batches:', len(train_pos_char_token_data))
print('Val batches:  ', len(val_pos_char_token_data))
print('Test batches: ', len(test_pos_char_token_data))

Train batches: 5627
Val batches:   945
Test batches:  942


## 10. Compile & Train

In [18]:
os.makedirs('saved_models', exist_ok=True)

model_5.compile(
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.2),
    optimizer=tf.keras.optimizers.Adam(),
    metrics=['accuracy']
)

checkpoint = tf.keras.callbacks.ModelCheckpoint(
    'saved_models/skimlit_model5_best.keras',
    monitor='val_accuracy', save_best_only=True, verbose=1
)
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_accuracy', patience=3, restore_best_weights=True
)
reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss', factor=0.2, patience=2, verbose=1
)

history_5 = model_5.fit(
    train_pos_char_token_data,                              # FIXED variable name
    steps_per_epoch=int(0.1 * len(train_pos_char_token_data)),
    validation_data=val_pos_char_token_data,               # FIXED variable name
    validation_steps=int(0.1 * len(val_pos_char_token_data)),
    epochs=10,
    callbacks=[checkpoint, early_stop, reduce_lr]
)

Epoch 1/10


562/562 ━━━━━━━━━━━━━━━━━━━━ 0s 67ms/step - accuracy: 0.6687 - loss: 1.1450
Epoch 1: val_accuracy improved from None to 0.85040, saving model to saved_models/skimlit_model5_best.keras

Epoch 1: finished saving model to saved_models/skimlit_model5_best.keras
562/562 ━━━━━━━━━━━━━━━━━━━━ 46s 72ms/step - accuracy: 0.7730 - loss: 1.0067 - val_accuracy: 0.8504 - val_loss: 0.8988 - learning_rate: 0.0010
Epoch 2/10
562/562 ━━━━━━━━━━━━━━━━━━━━ 0s 73ms/step - accuracy: 0.8583 - loss: 0.8995
Epoch 2: val_accuracy improved from 0.85040 to 0.86370, saving model to saved_models/skimlit_model5_best.keras

Epoch 2: finished saving model to saved_models/skimlit_model5_best.keras
562/562 ━━━━━━━━━━━━━━━━━━━━ 43s 77ms/step - accuracy: 0.8561 - loss: 0.8930 - val_accuracy: 0.8637 - val_loss: 0.8723 - learning_rate: 0.0010
Epoch 3/10
562/562 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step - accuracy: 0.8658 - loss: 0.8743
Epoch 3: val_accuracy improved from 0.86370 to 0.87168, saving model to saved_models/skimlit_mode

## 11. Save Model & Class Names

In [19]:
model_5.save('saved_models/skimlit_model5_final.keras')

with open('saved_models/class_names.json', 'w') as f:
    json.dump(class_names, f)

print('Model saved to saved_models/skimlit_model5_final.keras')
print('Class names saved to saved_models/class_names.json')

Model saved to saved_models/skimlit_model5_final.keras
Class names saved to saved_models/class_names.json


## 12. Evaluate on Validation & Test Sets

In [20]:
print('=== Validation Set ===')
model_5.evaluate(val_pos_char_token_data, steps=len(val_pos_char_token_data))

=== Validation Set ===
945/945 ━━━━━━━━━━━━━━━━━━━━ 14s 14ms/step - accuracy: 0.8936 - loss: 0.8242


[0.824174165725708, 0.8935522437095642]

In [21]:
print('=== Test Set Predictions ===')
preds = model_5.predict(test_pos_char_token_data)          # FIXED variable name
preds_probs = tf.argmax(preds, axis=1).numpy()
test_labels_encoded = np.array(test_labels_encoded)

=== Test Set Predictions ===
942/942 ━━━━━━━━━━━━━━━━━━━━ 15s 15ms/step


## 13. Results Calculator & Comparison

In [22]:
CLASS_NAMES = ['BACKGROUND', 'CONCLUSIONS', 'METHODS', 'OBJECTIVE', 'RESULTS']

def calculate_results(y_true, y_pred, model_name='Model'):
    if len(np.array(y_true).shape) > 1:
        y_true = np.argmax(y_true, axis=1)
    if len(np.array(y_pred).shape) > 1:
        y_pred = np.argmax(y_pred, axis=1)
    accuracy = accuracy_score(y_true, y_pred) * 100
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average='weighted'
    )
    return {
        'model':     model_name,
        'accuracy':  round(accuracy, 2),
        'precision': round(precision, 4),
        'recall':    round(recall, 4),
        'f1':        round(f1, 4),
    }

def per_class_report(y_true, y_pred, model_name='Model'):
    if len(np.array(y_true).shape) > 1:
        y_true = np.argmax(y_true, axis=1)
    if len(np.array(y_pred).shape) > 1:
        y_pred = np.argmax(y_pred, axis=1)
    print(f'\n{"="*55}')
    print(f'  Per-class report: {model_name}')
    print(f'{"="*55}')
    print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

model5_results = calculate_results(test_labels_encoded, preds_probs, 'SkimLit Model 5')
per_class_report(test_labels_encoded, preds_probs, 'SkimLit Model 5')
print('\nOverall:', model5_results)


  Per-class report: SkimLit Model 5
              precision    recall  f1-score   support

  BACKGROUND       0.73      0.88      0.80      3621
 CONCLUSIONS       0.94      0.90      0.92      4571
     METHODS       0.91      0.95      0.93      9897
   OBJECTIVE       0.83      0.51      0.63      2333
     RESULTS       0.92      0.92      0.92      9713

    accuracy                           0.89     30135
   macro avg       0.87      0.83      0.84     30135
weighted avg       0.89      0.89      0.89     30135


Overall: {'model': 'SkimLit Model 5', 'accuracy': 88.98, 'precision': 0.8918, 'recall': 0.8898, 'f1': 0.8869}


## 14. PubMedBERT Benchmark Comparison

In [23]:
from transformers import pipeline

CANDIDATE_MODELS = [
    'qanastek/PubMedBERT-NLI-RCT',
    'microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract',
    'allenai/scibert_scivocab_uncased',
]

pubmedbert = None
loaded_model_name = None

for model_id in CANDIDATE_MODELS:
    try:
        print(f'Trying {model_id}...')
        pubmedbert = pipeline('text-classification', model=model_id, device=0)
        loaded_model_name = model_id
        print(f'Loaded: {model_id} ✓')
        break
    except Exception as e:
        print(f'  Failed: {e}\n')

# Check raw label format before running full inference
sample = pubmedbert(test_sentences[0])
print('\nSample output:', sample)
print('Raw label:', sample[0]['label'])
print('\n⚠️  Check the label above and update PUBMEDBERT_LABEL_MAP below if needed')

Trying qanastek/PubMedBERT-NLI-RCT...


  Failed: qanastek/PubMedBERT-NLI-RCT is not a local folder and is not a valid model identifier listed on 'https://huggingface.co/models'
If this is a private repository, make sure to pass a token having permission to this repo either by logging in with `hf auth login` or by passing `token=<your_token>`

Trying microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract...
  Failed: name 'torch' is not defined

Trying allenai/scibert_scivocab_uncased...
  Failed: name 'torch' is not defined



TypeError: 'NoneType' object is not callable

In [ ]:
from transformers import pipeline
import torch 
CANDIDATE_MODELS = [
    "pritamdeka/PubMedBert-PubMed200kRCT",
    "pritamdeka/PubMedBERT-MNLI-MedNLI",
    "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract",
    "allenai/scibert_scivocab_uncased",
]

pubmedbert = None
loaded_model_name = None

for model_id in CANDIDATE_MODELS:
    try:
        print(f"Trying {model_id}...")
        pubmedbert = pipeline("text-classification", model=model_id, device=0)
        loaded_model_name = model_id
        print(f"Loaded: {model_id} ✓")
        break
    except Exception as e:
        print(f"  Failed: {e}\n")

if pubmedbert is None:
    raise RuntimeError("No model loaded. Install torch/transformers and try again.")

sample = pubmedbert(test_sentences[0])
print("\nSample output:", sample)
print("Raw label:", sample[0]["label"])

Trying pritamdeka/PubMedBert-PubMed200kRCT...
  Failed: name 'torch' is not defined

Trying pritamdeka/PubMedBERT-MNLI-MedNLI...
  Failed: name 'torch' is not defined

Trying microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract...
  Failed: name 'torch' is not defined

Trying allenai/scibert_scivocab_uncased...
  Failed: name 'torch' is not defined



RuntimeError: No model loaded. Install torch/transformers and try again.

In [ ]:
# ── Update this map to match whatever label format the model returns ─────
# Common formats: 'BACKGROUND', 'background', 'LABEL_0'
# Run the cell above first, then adjust if needed
PUBMEDBERT_LABEL_MAP = {
    'BACKGROUND':  0,
    'CONCLUSIONS': 1,
    'METHODS':     2,
    'OBJECTIVE':   3,
    'RESULTS':     4,
    # lowercase fallbacks
    'background':  0,
    'conclusions': 1,
    'methods':     2,
    'objective':   3,
    'results':     4,
}

print('Running PubMedBERT inference on test set (this may take ~10 min)...')
BATCH_SIZE_BERT = 64
bert_preds = []

for i in range(0, len(test_sentences), BATCH_SIZE_BERT):
    batch   = test_sentences[i : i + BATCH_SIZE_BERT]
    results = pubmedbert(batch, truncation=True, max_length=512)
    for r in results:
        label = r['label']
        bert_preds.append(PUBMEDBERT_LABEL_MAP.get(label, 0))
    if i % 5000 == 0:
        print(f'  {i}/{len(test_sentences)} done...')

bert_preds   = np.array(bert_preds)
bert_results = calculate_results(test_labels_encoded, bert_preds, 'PubMedBERT')
per_class_report(test_labels_encoded, bert_preds, 'PubMedBERT')
print('\nOverall:', bert_results)

## 15. Final Comparison Table

In [ ]:
baseline_results = {
    'model':     'Baseline (TF-IDF)',
    'accuracy':  72.18,
    'precision': 0.7186,
    'recall':    0.7218,
    'f1':        0.6989,
}

comparison_df = pd.DataFrame([baseline_results, model5_results, bert_results])
comparison_df = comparison_df.set_index('model')

print('\n' + '='*55)
print('  FINAL COMPARISON TABLE')
print('='*55)
print(comparison_df.to_string())

print('\n  Best per metric:')
for col in ['accuracy', 'precision', 'recall', 'f1']:
    best = comparison_df[col].idxmax()
    print(f'  {col:12} -> {best}  ({comparison_df.loc[best, col]})')

## 16. Save Results & Download

In [ ]:
all_results = {
    'baseline':   baseline_results,
    'model_5':    model5_results,
    'pubmedbert': bert_results,
}

with open('saved_models/comparison_results.json', 'w') as f:
    json.dump(all_results, f, indent=2)

print('Results saved to saved_models/comparison_results.json ✓')
print('\nScreenshot the comparison table above — those are your resume numbers.')

In [ ]:
# Download everything from Colab
import shutil
from google.colab import files

shutil.make_archive('saved_models', 'zip', 'saved_models')
files.download('saved_models.zip')
print('Download started ✓')